# Edge Data Preprocessing Pipeline 

---

##  1. 資料感測器輸入 (Sensor Data Dictionary)
本系統採集塗裝產線的流體控制、機器手臂運動學與 CCD 視覺品質數據。採樣頻率設定為 **1Hz (每秒一筆)**。

| 項目 (Item) | 系統欄位名稱 (Internal) | 單位 (Unit) | 格式 (Format) | 物理位置/關聯 (Location / Relation) |
| :--- | :--- | :--- | :--- | :--- |
| 批次編號 | `batch_id` | - | string | System (對齊 50 秒噴塗週期主鍵) |
| 資料時間戳 | `timestamp` | ISO8601 | string | Edge_Gateway (邊緣閘道器絕對時間) |
| 作業站點 | `station` | - | string | System (Station_1, Station_2, Station_3) |
| 膜厚度 | `film_thickness_um` | μm | float | Quality_Sensor (最終品質 Y 值) |
| 塗料流量 | `paint_flow_ml_min` | ml/min | float | Fluid_Controller |
| 空氣壓力 | `air_pressure_bar` | bar | float | Fluid_Controller |
| 濾網壓差 | `filter_diff_pressure_bar` | bar | float | Fluid_Pipeline (預測性維護 AI 核心目標)|
| 噴幅寬度 | `spray_width_mm` | mm | float | CCD_Vision_System |
| 伺服馬達負載 | `servo_torque_load_pct` | % | float | Robot_Controller |
| 減速機溫度 | `gearbox_temperature_c` | °C | float | Robot_Joint |
| 軌跡追隨誤差 | `path_error_mm` | mm | float | Robot_Controller |
| 三軸振動加速度 | `vibration_g` | G | float | Robot_Base |
| 環境溫度 | `temperature_c` | °C | float | Chamber_Env |
| 環境濕度 | `humidity_rh` | % | float | Chamber_Env |
| TCP 空間座標 | `tcp_x_mm`, `y`, `z` | mm | float | Robot_Arm  |
| 噴嘴翻滾角 | `nozzle_roll` | degree | float | Robot_Arm |
| 移動速度 | `speed_mm_s` | mm/s | float | Robot_Arm |

---

##  2. 感測器資料標準：正常與異常值定義 (Thresholds)
為建立 AI 預測壽命 (RUL) 的目標，並提供 UI 戰情室各區塊的警報邏輯，系統針對所有連續數值型感測器，定義了以下工業標準：

### 【群組一】預測性維護 (PdM) 核心衰退指標
這組特徵會隨時間與使用次數逐漸惡化，是 AI 預測「剩餘有用壽命 (RUL)」的主要 Target。

#### A. 塗料濾網壓差 (`filter_diff_pressure_bar`)
定義流體通過金屬濾網時產生的阻力。隨天數累積漆渣，壓差會逐漸上升：
* 🟢 **正常 (OK)：`0.15 ~ 0.30 bar`** 
* 🟡 **預警 (Warning)：`0.31 ~ 0.50 bar`** 
* 🔴 **異常/停機 (Bad)：`> 0.50 bar`** 

#### B. 伺服馬達負載 (`servo_torque_load_pct`)
定義機械手臂關節馬達的出力百分比。隨減速機磨損與潤滑油老化而逐漸增加：
* 🟢 **正常 (OK)：`< 60 %`** (阻力極小，運行順暢)
* 🟡 **預警 (Warning)：`60 ~ 80 %`** (摩擦力增加，馬達需額外出力)
* 🔴 **異常 (Bad)：`> 80 %`** (過載風險極高，可能導致馬達燒毀)

---

### 【群組二】品質與流體控制特徵
這組特徵主要用於監控當下的塗裝品質，通常維持在固定範圍，若發生偏移即代表製程參數跑掉。

#### C. CCD 視覺噴幅寬度 (`spray_width_mm`)
採用**「基礎值 ± 絕對公差」**作為動態判定標準，適配不同站點的噴嘴。
* **各站基礎值 (Base)：** M1 (120mm) / M2 (100mm) / M3 (82mm)
* 🟢 **正常 (OK)：** 誤差 `≤ ±10 mm` (塗布均勻)
* 🟡 **預警 (Warning)：** 誤差介於 `±11 mm ~ ±20 mm` (可能造成邊緣漏噴或局部過厚)
* 🔴 **異常 (Bad)：** 誤差 `> ±20 mm` (嚴重積漆或霧化壓力消失)

#### D. 膜厚度 (`film_thickness_um`)
定義最終塗裝成品的漆膜厚度。目標基準值為 15.0 μm：
* 🟢 **正常 (OK)：`14.5 ~ 15.5 μm`** (符合品質允收標準)
* 🟡 **預警 (Warning)：`14.0 ~ 14.4 μm` 或 `15.6 ~ 16.0 μm`** (品管邊緣，需微調參數)
* 🔴 **異常 (Bad)：`< 14.0 μm` 或 `> 16.0 μm`** (膜厚不足易透底，過高易垂流)

#### E. 塗料流量 (`paint_flow_ml_min`)
定義齒輪幫浦每分鐘輸出的塗料量。目標基準值為 250.0 ml/min：
* 🟢 **正常 (OK)：`245.0 ~ 255.0 ml/min`**
* 🟡 **預警 (Warning)：`235.0 ~ 244.9 ml/min` 或 `255.1 ~ 265.0 ml/min`** (幫浦些微磨損或管路微塞)
* 🔴 **異常 (Bad)：`< 235.0 ml/min` 或 `> 265.0 ml/min`** (嚴重供料異常)

---

### 【群組三】設備即時防護特徵 (防撞與環境)
對瞬間變化極度敏感，通常交由 OT 端 (PLC) 進行毫秒級停機，並將事件上傳 IT 端。

#### F. 三軸振動加速度 (`vibration_g`)
定義機械手臂底座或末端感測到的震動：
* 🟢 **正常 (OK)：`< 1.5 G`** (正常運行微震)
* 🟡 **預警 (Warning)：`1.5 ~ 3.0 G`** (軌道不平整或機台共振)
* 🔴 **異常/防撞 (Bad)：`> 3.0 G`** (極可能發生實體碰撞或重大機械卡死)

#### G. 軌跡追隨誤差 (`path_error_mm`)
定義實際 TCP 座標與目標座標的偏差值：
* 🟢 **正常 (OK)：`< 0.5 mm`**
* 🟡 **預警 (Warning)：`0.5 ~ 1.5 mm`** (減速機背隙變大，易產生漆面斑馬紋)
* 🔴 **異常 (Bad)：`> 1.5 mm`** (嚴重失位，需重新校正 TCP)

#### H. 減速機溫度 (`gearbox_temperature_c`)
定義機械手臂主要關節的運行溫度：
* 🟢 **正常 (OK)：`< 55 °C`**
* 🟡 **預警 (Warning)：`55 ~ 70 °C`** (連續運行溫升，建議提高散熱)
* 🔴 **異常 (Bad)：`> 70 °C`** (散熱失效或嚴重摩擦，強制降速)

#### I. 環境溫濕度 (`temperature_c` / `humidity_rh`)
定義無塵噴漆室的環境控制：
* 🟢 **正常 (OK)：`22 ~ 26 °C` / `50 ~ 60 %RH`** (最佳揮發與附著條件)
* 🔴 **異常 (Bad)：** 超出此範圍即發出環境警報 (易導致橘皮、白化等烤漆缺陷)

---

##  3. 核心處理函式 (Core Functions)

本專案將資料管線封裝為以下核心函式與類別：

1. **`generate_ultimate_1Hz_data_with_pdm()`**
   * **作用：** 數位孿生資料生成器。負責模擬連續 10 天的產線運作，並精準注入隨機雜訊 (NaN、突波) 以及時間老化的因果特徵 (如 `wear_factor` 造成的壓差上升與噴嘴積漆)。
2. **`class DataPreprocessingService`**
   * **作用：** 邊緣運算清洗微服務的主體類別 (Class)，定義了需要清洗的特徵陣列與中介資料。
3. **`process_single_batch(df_batch)`**
   * **作用：** 單批次清洗核心。強制以 `batch_id` (50秒) 為單位進行運算，避免跨時段 (如中午休息) 產生錯誤的數學插值。
4. **`run_pipeline(file_path)`**
   * **作用：** ETL 管線入口。負責讀取原始檔案、呼叫分批清洗邏輯、重新排序時間軸，並回傳乾淨的 DataFrame。

---

##  4. 邊緣清洗演算法 (Data Cleaning Algorithms)
針對工業現場常見的「感測器斷線」與「電磁干擾/物理突波」，微服務內建三階段清洗策略：

1. **空值修補 (Missing Value Imputation)：**
   * 演算法：**線性插值 (`linear interpolation`) + 前後填充 (`ffill / bfill`)**。
   * 目的：修補 NaN，確保時間序列資料的波形連續性，避免 AI 模型訓練報錯。
2. **極端突波濾除 (Outlier Handling)：**
   * 演算法：**四分位距法 (IQR, Interquartile Range)**。
   * 目的：針對 `servo_torque_load_pct`、`vibration_g` 等易受干擾的特徵，計算 `Q1 - 1.5*IQR` 與 `Q3 + 1.5*IQR` 的合理邊界。將超出邊界的極端突發值強制設為 `NaN`，並再次呼叫線性插值抹平。
3. **訊號去噪與平滑化 (Signal Smoothing & Feature Expansion)：**
   * 演算法：**5 秒移動平均 (5-second Rolling Mean)**。
   * 目的：針對所有連續物理量進行水平特徵擴增 (Horizontal Concatenation)，生成如 `vibration_g_smooth` 的新欄位，提供 AI 更容易抓取長期衰退斜率的「趨勢線」。

---

##  5. 系統輸出架構 (System Outputs)
秉持「冷熱資料分層 (Tiered Storage)」與「Payload 瘦身」架構原則，一鍵執行腳本後，每個站點 (Station 1~3) 皆會自動產出 3 份檔案（全產線共產出 9 份），並精準分派給下游團隊：

| 產出檔案名稱後綴 | 檔案格式 | 資料層級 | 下游接收方 | 系統設計意義 |
| :--- | :--- | :--- | :--- | :--- |
| **`_Raw (原始生成)`** | CSV |  冷資料 |  | 保留在本地資料湖泊 (Data Lake)，紀錄最真實的物理碰撞與斷線，供未來開發「異常偵測」AI 使用。 |
| **`_Cleaned_ReadyForAI`** | CSV |  熱資料 | AI 預測組 (沈/少) | 包含原始洗淨特徵 + `_smooth` 平滑特徵。讓 AI 同時掌握瞬間爆發異常與長期設備老化雙重視角。 |
| **`_Cleaned_Full`** | JSON |  備份資料 |  | 完整的 JSON Payload 備份，供 API 除錯使用。 |
| **`_Cleaned_SlimForDB`** | JSON |  熱資料 | DB | **[Payload 瘦身]**：剔除原始震盪特徵，僅保留 Metadata 與平滑化數值 (並移除 smooth 後綴以適配 DB Schema)。大幅降低傳輸頻寬與資料庫負載，確保戰情室渲染順暢。 |

In [ ]:

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def generate_ultimate_1Hz_data_with_pdm():
    days = 10
    rows_per_batch = 50  # 1Hz 採樣，50秒產生 50 筆
    
    shifts = [
        {"name": "Morning", "start_hour": 8, "end_hour": 12},
        {"name": "Afternoon", "start_hour": 13, "end_hour": 17}
    ]
    
    # 站點設定 (包含膜厚、流量、壓力與 CCD 視覺噴幅基準)
    stations = {
        "Station_1": {"file": "Station_1_Base_1Hz_PdM.csv", "thick": 15.0, "flow": 250.0, "press": 3.5, "width": 120.0},
        "Station_2": {"file": "Station_2_Color_1Hz_PdM.csv", "thick": 15.0, "flow": 250.0, "press": 3.2, "width": 100.0},
        "Station_3": {"file": "Station_3_Clear_1Hz_PdM.csv", "thick": 15.0, "flow": 250.0, "press": 3.0, "width": 82.0}
    }
    
    columns = [
        "batch_id", "timestamp", "station", 
        "film_thickness_um", "paint_flow_ml_min", "air_pressure_bar", 
        "filter_diff_pressure_bar", "spray_width_mm",  # 兩大黃金特徵 (濾網與噴幅)
        "servo_torque_load_pct", "gearbox_temperature_c", "path_error_mm", "vibration_g",
        "temperature_c", "humidity_rh",
        "tcp_x_mm", "tcp_y_mm", "tcp_z_mm", "nozzle_roll", "speed_mm_s"
    ]
    
    for st_info in stations.values():
        pd.DataFrame(columns=columns).to_csv(st_info["file"], index=False)
        
    start_date = datetime.strptime("2026-05-28", "%Y-%m-%d")
    total_expected_batches = days * 8 * 60
    total_batches_generated = 0

    print("🚀 啟動 [AI預測維護版 1Hz] 數位孿生數據生成器...")
    print("🎯 已載入濾網壓差 (ΔP) 黃金標準：正常<0.30 | 預警>0.30 | 異常>0.50 bar")
    
    for day in range(days):
        current_date = start_date + timedelta(days=day)
        
        for shift in shifts:
            st1_buffer, st2_buffer, st3_buffer = [], [], []
            
            for hour in range(shift["start_hour"], shift["end_hour"]):
                for minute in range(60):
                    batch_id = f"B_{current_date.strftime('%Y%m%d')}_{hour:02d}{minute:02d}"
                    
                    t1_start = current_date.replace(hour=hour, minute=minute, second=0, microsecond=0)
                    t2_start = t1_start + timedelta(seconds=170)
                    t3_start = t2_start + timedelta(seconds=170)
                    
                    # 設備老化因子 (0.0 -> 1.0)
                    wear_factor = total_batches_generated / total_expected_batches
                    
                    for st_name, t_start in zip(["Station_1", "Station_2", "Station_3"], [t1_start, t2_start, t3_start]):
                        
                        timestamps = [(t_start + timedelta(seconds=1*i)).strftime("%Y-%m-%dT%H:%M:%SZ") for i in range(rows_per_batch)]
                        
                        st_params = stations[st_name]
                        base_width = st_params["width"]
                        
                        thickness = np.random.normal(st_params["thick"], 0.2, rows_per_batch)
                        flow = np.random.normal(st_params["flow"], 1.0, rows_per_batch)
                        pressure = np.random.normal(st_params["press"], 0.02, rows_per_batch)
                        
                        # 【核心修改：濾網精準老化模擬】
                        # 第1天: 0.15 bar (全新)
                        # 第5天: ~0.35 bar (進入黃色預警)
                        # 第9天: ~0.51 bar (突破紅色異常線，AI 預測達標！)
                        filter_diff = np.random.normal(0.15 + (wear_factor * 0.4), 0.01, rows_per_batch)
                        
                        temp = np.random.normal(24.5, 0.1, rows_per_batch)
                        humidity = np.random.normal(55.2, 0.5, rows_per_batch)
                        
                        time_steps = np.arange(rows_per_batch)
                        tcp_x_mm = 125.0 + np.sin(time_steps * 0.2) * 50
                        tcp_y_mm = 200.0 + np.cos(time_steps * 0.1) * 30
                        tcp_z_mm = np.full(rows_per_batch, 50.2)
                        nozzle_roll = np.full(rows_per_batch, 45.0)
                        speed_mm_s = np.full(rows_per_batch, 20.0)
                        
                        # Station 2 機器手臂老化與噴嘴積漆現象
                        if st_name == "Station_2":
                            torque = np.random.normal(50.0 + (wear_factor * 25.0), 1.0, rows_per_batch) 
                            gearbox_temp = np.random.normal(40.0 + (wear_factor * 15.0), 0.2, rows_per_batch)
                            path_err = np.abs(np.random.normal(0.1 + (wear_factor * 0.3), 0.02, rows_per_batch))
                            ccd_width = np.random.normal(base_width - (wear_factor * 8.0), 1.0 + (wear_factor * 2.0), rows_per_batch)
                        else:
                            torque = np.random.normal(50.0, 1.0, rows_per_batch)
                            gearbox_temp = np.random.normal(40.0, 0.2, rows_per_batch)
                            path_err = np.abs(np.random.normal(0.1, 0.01, rows_per_batch))
                            ccd_width = np.random.normal(base_width, 1.0, rows_per_batch)
                            
                        vibration = np.random.normal(0.8, 0.05, rows_per_batch)
                        
                        df_batch = pd.DataFrame({
                            "batch_id": batch_id, "timestamp": timestamps, "station": st_name,
                            "film_thickness_um": np.round(thickness, 2), "paint_flow_ml_min": np.round(flow, 2),
                            "air_pressure_bar": np.round(pressure, 2), 
                            "filter_diff_pressure_bar": np.round(filter_diff, 3), 
                            "spray_width_mm": np.round(ccd_width, 1),
                            "servo_torque_load_pct": np.round(torque, 1), "gearbox_temperature_c": np.round(gearbox_temp, 1), 
                            "path_error_mm": np.round(path_err, 3), "vibration_g": np.round(vibration, 2), 
                            "temperature_c": np.round(temp, 2), "humidity_rh": np.round(humidity, 2),
                            "tcp_x_mm": np.round(tcp_x_mm, 2), "tcp_y_mm": np.round(tcp_y_mm, 2),
                            "tcp_z_mm": np.round(tcp_z_mm, 2), "nozzle_roll": np.round(nozzle_roll, 1),
                            "speed_mm_s": np.round(speed_mm_s, 1)
                        })
                        
                        # 空值與突波注入 (考驗你的清洗微服務)
                        if np.random.rand() < 0.2: 
                            nan_idx = np.random.choice(rows_per_batch, size=1)
                            df_batch.loc[nan_idx, 'filter_diff_pressure_bar'] = np.nan
                            
                        if st_name == "Station_1": st1_buffer.append(df_batch)
                        elif st_name == "Station_2": st2_buffer.append(df_batch)
                        else: st3_buffer.append(df_batch)
                    
                    total_batches_generated += 1
            
            # 寫入 CSV
            pd.concat(st1_buffer, ignore_index=True).to_csv(stations["Station_1"]["file"], mode='a', header=False, index=False)
            pd.concat(st2_buffer, ignore_index=True).to_csv(stations["Station_2"]["file"], mode='a', header=False, index=False)
            pd.concat(st3_buffer, ignore_index=True).to_csv(stations["Station_3"]["file"], mode='a', header=False, index=False)
            
        print(f"✅ 第 {day+1}/10 天生成完畢。")
    print("🎉 3 份終極完美版資料集生成完畢！請交給各組進行開發！")

# 執行
generate_ultimate_1Hz_data_with_pdm()

🚀 啟動 [AI預測維護版 1Hz] 數位孿生數據生成器...
🎯 已載入濾網壓差 (ΔP) 黃金標準：正常<0.30 | 預警>0.30 | 異常>0.50 bar
✅ 第 1/10 天生成完畢。
✅ 第 2/10 天生成完畢。
✅ 第 3/10 天生成完畢。
✅ 第 4/10 天生成完畢。
✅ 第 5/10 天生成完畢。
✅ 第 6/10 天生成完畢。
✅ 第 7/10 天生成完畢。
✅ 第 8/10 天生成完畢。
✅ 第 9/10 天生成完畢。
✅ 第 10/10 天生成完畢。
🎉 3 份終極完美版資料集生成完畢！請交給各組進行開發！


In [7]:
import pandas as pd
import numpy as np
import json
import os

class DataPreprocessingService:
    def __init__(self):
        # 基礎 Metadata
        self.metadata_cols = ['batch_id', 'timestamp', 'station']
        
        # 需平滑化與修補的連續數值
        self.sensor_cols = [
            'film_thickness_um', 'paint_flow_ml_min', 'air_pressure_bar',
            'filter_diff_pressure_bar', 'spray_width_mm', 
            'servo_torque_load_pct', 'gearbox_temperature_c', 'path_error_mm', 'vibration_g',
            'temperature_c', 'humidity_rh',
            'tcp_x_mm', 'tcp_y_mm', 'tcp_z_mm', 'nozzle_roll', 'speed_mm_s'
        ]
        
        # 需用 IQR 濾除突波的特定特徵
        self.outlier_targets = ['servo_torque_load_pct', 'vibration_g', 'spray_width_mm']

    def process_single_batch(self, df_batch: pd.DataFrame) -> pd.DataFrame:
        df = df_batch.copy()
        
        # 1. 處理空值 (線性插值)
        df[self.sensor_cols] = df[self.sensor_cols].interpolate(method='linear', limit_direction='both')
        df[self.sensor_cols] = df[self.sensor_cols].ffill().bfill()
        
        # 2. 處理突波 (IQR)
        for col in self.outlier_targets:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
            df.loc[outlier_mask, col] = np.nan
            df[col] = df[col].interpolate(method='linear', limit_direction='both').ffill().bfill()
            
        # 3. 訊號平滑化 (5秒滑動平均)
        for col in self.sensor_cols:
            smooth_col_name = f"{col}_smooth"
            df[smooth_col_name] = df[col].rolling(window=5, min_periods=1).mean()
            
        return df

    def run_pipeline(self, file_path: str):
        df_raw = pd.read_csv(file_path)
        # 批次清洗
        df_clean = df_raw.groupby('batch_id', group_keys=False).apply(self.process_single_batch)
        df_clean = df_clean.sort_values(by='timestamp').reset_index(drop=True)
        return df_clean

# ==========================================
# 🚀 主程式：一鍵批次處理三站資料
# ==========================================
if __name__ == "__main__":
    preprocessor = DataPreprocessingService()
    
    # 定義要讀取的 3 份原始檔案名稱與輸出前綴
    stations_to_process = [
        {"raw_file": "Station_1_Base_1Hz_PdM.csv", "prefix": "Station_1_Base"},
        {"raw_file": "Station_2_Color_1Hz_PdM.csv", "prefix": "Station_2_Color"},
        {"raw_file": "Station_3_Clear_1Hz_PdM.csv", "prefix": "Station_3_Clear"}
    ]
    
    print("🌟 開始執行全產線自動化清洗與分派腳本...\n")
    
    for st in stations_to_process:
        raw_file = st["raw_file"]
        prefix = st["prefix"]
        
        if not os.path.exists(raw_file):
            print(f"❌ 找不到檔案 {raw_file}，請確認檔名或路徑。")
            continue
            
        print(f"⚙️ 正在處理: {raw_file}")
        
        # 1. 執行清洗演算法
        df_clean = preprocessor.run_pipeline(raw_file)
        
        # 2. 產出物 A：給 AI 組的 CSV (全欄位)
        csv_filename = f"{prefix}_Cleaned_ReadyForAI.csv"
        df_clean.to_csv(csv_filename, index=False)
        
        # 3. 產出物 B：完整的 JSON (Full payload)
        df_clean_json = df_clean.replace({float('nan'): None})
        full_json_filename = f"{prefix}_Cleaned_Full.json"
        with open(full_json_filename, 'w', encoding='utf-8') as f:
            json.dump(df_clean_json.to_dict(orient='records'), f, ensure_ascii=False, indent=4)
            
        # 4. 產出物 C：給資料庫的瘦身 JSON (Slim payload)
        base_cols = ['batch_id', 'timestamp', 'station']
        smooth_cols = [col for col in df_clean.columns if str(col).endswith('_smooth')]
        df_for_db = df_clean[base_cols + smooth_cols].copy()
        
        # 把欄位名稱的 _smooth 拿掉，對接資料庫 Schema
        df_for_db.columns = [col.replace('_smooth', '') for col in df_for_db.columns]
        df_for_db = df_for_db.replace({float('nan'): None})
        
        slim_json_filename = f"{prefix}_Cleaned_SlimForDB.json"
        with open(slim_json_filename, 'w', encoding='utf-8') as f:
            json.dump(df_for_db.to_dict(orient='records'), f, ensure_ascii=False, indent=4)
            
        print(f"   ✅ 成功匯出: {csv_filename}")
        print(f"   ✅ 成功匯出: {full_json_filename}")
        print(f"   ✅ 成功匯出: {slim_json_filename}\n")

    print("🎉 全產線資料清洗與分派完成！你現在擁有所有站點的最完美資料了！")

🌟 開始執行全產線自動化清洗與分派腳本...

⚙️ 正在處理: Station_1_Base_1Hz_PdM.csv


C:\Users\ASUS\AppData\Local\Temp\ipykernel_27472\4058285486.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean = df_raw.groupby('batch_id', group_keys=False).apply(self.process_single_batch)


   ✅ 成功匯出: Station_1_Base_Cleaned_ReadyForAI.csv
   ✅ 成功匯出: Station_1_Base_Cleaned_Full.json
   ✅ 成功匯出: Station_1_Base_Cleaned_SlimForDB.json

⚙️ 正在處理: Station_2_Color_1Hz_PdM.csv


C:\Users\ASUS\AppData\Local\Temp\ipykernel_27472\4058285486.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean = df_raw.groupby('batch_id', group_keys=False).apply(self.process_single_batch)


   ✅ 成功匯出: Station_2_Color_Cleaned_ReadyForAI.csv
   ✅ 成功匯出: Station_2_Color_Cleaned_Full.json
   ✅ 成功匯出: Station_2_Color_Cleaned_SlimForDB.json

⚙️ 正在處理: Station_3_Clear_1Hz_PdM.csv


C:\Users\ASUS\AppData\Local\Temp\ipykernel_27472\4058285486.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean = df_raw.groupby('batch_id', group_keys=False).apply(self.process_single_batch)


   ✅ 成功匯出: Station_3_Clear_Cleaned_ReadyForAI.csv
   ✅ 成功匯出: Station_3_Clear_Cleaned_Full.json
   ✅ 成功匯出: Station_3_Clear_Cleaned_SlimForDB.json

🎉 全產線資料清洗與分派完成！你現在擁有所有站點的最完美資料了！
